# General

## Imports

In [ ]:
### Time frames - Define the start and end dates of the data that you want to look at (e.g. all 29 years between 1996 and 2025) (ICMECAT only goes to mid-2025 for now)
dataTimeframe = ["1996-01-01 00:00", "2025-01-01 00:00"]
dataSourcesToUse = ["Wind"]

import numpy as np 
import os
import pandas as pd
import calendar
import warnings
warnings.filterwarnings("ignore")

## Functions

In [ ]:
def Yearly(startDate, endDate, frequency = 'YS'): #ReadDatesForFileFinding - Gets range of dates in given frequency to use to get the data files.
    YearlyRange = pd.date_range(startDate, endDate, freq=frequency).strftime("%Y%m%d").tolist()
    return YearlyRange

# Get to date portion of the file names
def myFunc(e):
  return e.split("b")[1]

# Wind: Get the pre-downloaded data from folder
def FindDataWind(directory, dataName, startDate, endDate, frequency = 'MS'): #FindDataMakeTimeSeries
    files = []
    for file in os.listdir(directory):
        filename = os.fsdecode(file)
        DateRange = pd.date_range(startDate, endDate, freq=frequency).strftime("%Y%m%d").tolist()
        for j in range(len(DateRange)):
            if filename.startswith(dataName + str(DateRange[j])[0:6]):
                files.append(directory+filename)
                continue
    filesSorted = sorted(files, key=myFunc) # sort files
    wind_list = [] #pd.DataFrame([])
    # Columns from https://omniweb.gsfc.nasa.gov/html/HROdocum.html
    colnames = ["Year", "DOY", "Hour", "Minute", "IMF_PTS", "PercentInterp", "CP/MVFlag", "Timeshift", 
                "PFN_XGSE", "PFN_YGSE","PFN_ZGSE", "ScalarB", "Bx_GSEGSM", "ByGSE", "BzGSE", "ByGSM", "BzGSM", 
                "RMSTimeshift", "RMS_PFN", "RMSScalarB", "RMSFieldVector", "Plasma_PTS", "FlowSpeed",
                "Vx_GSE", "Vy_GSE", "Vz_GSE", "ProtonDensity", "Temp", "X_s_c_GSE", "Y_s_c_GSE", "Z_s_c_GSE",
                 "X_targ_GSE", "Y_targ_GSE", "Z_targ_GSE", "RMS_targ", "DBOT1", "DBOT2"]
    for j in range(len(filesSorted)):
        wind_list.append( pd.read_csv(filesSorted[j], header=None, sep = r"\s+", names = colnames) )
    wind_monthly = pd.concat(wind_list, ignore_index=True)
    # Laura's code
    wind_monthly.index = (pd.to_datetime(wind_monthly["Year"].astype(str), format="%Y")
                + pd.to_timedelta(wind_monthly["DOY"] - 1, unit="D")
                + pd.to_timedelta(wind_monthly["Hour"], unit="h")
                + pd.to_timedelta(wind_monthly["Minute"], unit="min"))
    fill_values = {"ScalarB": 9999.99, "Bx_GSEGSM": 9999.99, "ByGSE": 9999.99, "BzGSE": 9999.99,
                   "FlowSpeed": 99999.9, "ProtonDensity": 999.99, "Temp": 9999999.0}
    for column, fill in fill_values.items():
        wind_monthly.loc[wind_monthly[column] >= fill, column] = np.nan
    return wind_monthly


# Wind

In [ ]:
### Wind
if "Wind" in dataSourcesToUse:
    #### Places to get and save files
    DataDirectory = "Data/windData/"
    dataName = "wind_min_b"
    ##### Details: Yearly, minutely data, bow-shock shifted

    # Get data files
    windData = [] # empty array for wind data
    start = dataTimeframe[0] # start date of data
    end = dataTimeframe[1] # end date of data

    yearlyRange = Yearly(start, end, frequency = 'MS') # Get range of dates needed (monthly as the downloaded data files are monthly)
    print(yearlyRange)
    for i in range(len(yearlyRange)-1): # for each date range in time range needed
        StartDate = yearlyRange[i] # first file in range
        dow, numberOfDays = calendar.monthrange(int(yearlyRange[i][0:4]), int(yearlyRange[i][4:6])) #dow = day of week, not needed value
        EndDate = yearlyRange[i][0:6] + str(numberOfDays) # last file in range 
        dataFound = FindDataWind(DataDirectory, dataName, StartDate, EndDate) # find data
        windData.append(dataFound) # add each month of data to the data array

    list_of_monthly_data = [] # create a list for the data frames
    for i in range(len(windData)): #for each monthly timeseries
        list_of_monthly_data.append(windData[i]) # add them to the list as dataframes
    windDataConcat = pd.concat(list_of_monthly_data) # concatenate all the monthly dataframes together to make one large dataframe for the full data timespan

else:
    print("Not processing Wind data")


In [ ]:
pd.show_versions()

In [ ]:
### Wind
if "Wind" in dataSourcesToUse:
    print(windDataConcat)
    windDataConcat.to_csv("Wind_" + str(start)[0:10] + "-" + str(end)[0:10]+ "InputOutput.csv")
else:
    print("Not processing Wind data")

### Wind test

In [ ]:
testData = pd.read_csv("Wind_" + str(start)[0:11] + "-" + str(end)[0:11]+ "_InputOutput.csv", index_col="DateTime")

In [ ]:
# Test that the full data range is plotted
testData["FlowSpeed"].plot() # Note this is not the total solar wind speed that we will use, this is just to test the data has loaded correctly

# x = (2026-1996) * 365.25 * 24 * 60
# print(x)